### nxsut generator — v3.0

MARIO-native pipeline: parses EXIOBASE Hybrid v3.3.18, updates the electricity supply mix from EMBER (via the nxbase query API), pools electricity trade behind a supply/need pass-through layer, and updates the trade mix from the **open ENTSO-E** scheduled-exchange set (via the nxbase query API). Fully open input chain — the first publishable nxsut version. Supersedes the retired `v2.1` (which used the proprietary Electricity Maps mix on the same MARIO-native pipeline).

ENTSO-E covers the European countries; every other EXIOBASE region (US, CN, JP, … and the RoW aggregates) is filled domestic-only — as are the origin-only regions ENTSO-E returns as an all-zero destination column (e.g. Luxembourg, Malta: their real import dependence disappears in this version, a known residual pending an ENTSO-E control-area fetch).

Set `user` and `year` in the first cell, then run top to bottom.

In [ ]:
import mario
import yaml
import os

with open('paths.yml', 'r') as file:  # open the yml file
    paths = yaml.safe_load(file)

user = 'LR'   # change this to your username
year = 2025   # change this to the year you want to build

paths = paths[user]
import warnings
warnings.filterwarnings("ignore")

Parse the raw EXIOBASE database and aggregate electricity to EMBER resolution. `meta.source` enables MARIO's EXIOBASE Rest-of-World member-country expansion when using EMBER.

In [ ]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')
db.meta.source = 'EXIOBASE Hybrid 3.3.18'
db.aggregate("support/aggregate_ee.xlsx", ignore_nan=True)

Supply mix from nxbase (query API) — see `support/nxbase_client.py`. MARIO reads the reduced EMBER snapshot from a transient file, regenerated every run.

In [ ]:
from support import nxbase_client as nxc

nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)
print(nxc.get_provenance(nxbase_api, ['EMBER Yearly Electricity Data 2025']))

ember_snapshot_path = 'support/_nxbase_ember_snapshot.csv'
nxc.get_ember_snapshot(nxbase_api).to_csv(ember_snapshot_path, index=False)

db.update_supply_mix(
    "electricity",
    scenario = 'baseline',
    year = year,
    ember_path = ember_snapshot_path,
)

Pool the trade of the selected commodities. MARIO adds the `" supply"` / `" need"` pass-through layer and stores the observed trade shares in the supply block market shares. The suffixes match the `NXS2` namespace rows in nxbase.

In [ ]:
traded_commodities = ['Electricity']
db.pool_trade(traded_commodities, supply_suffix=" supply", need_suffix=" need")

**Update trade mixes** from the open ENTSO-E scheduled-exchange set (nxbase query API). One origins-by-destinations matrix per commodity; a positive column sum — not mere presence — decides "covered" (ENTSO-E returns some destinations as an all-zero column, which `update_trade_mix` would otherwise reject). `rescale=True` normalizes each destination mix while preserving destination column totals.

In [ ]:
scenario = 'entsoe_trades'
if scenario not in db.scenarios:
    db.clone_scenario('baseline', scenario)

regions = list(db.get_index('Region'))
for commodity in traded_commodities:
    pooled = db.meta.pooled_trade_map[commodity]
    trades = nxc.get_trade_matrix(
        nxbase_api, year=year, commodity=commodity,
        source=f"ENTSO-E electricity import mix {year}",
    )
    trade_dict = {}
    for dest in regions:
        col = trades[dest] if dest in trades.columns else None
        trade_dict[dest] = (
            col.dropna().to_dict() if (col is not None and col.sum() > 0) else {dest: 1.0}
        )
    db.update_trade_mix(
        trade_dict,
        items = pooled['supply'],
        commodities = pooled['need'],
        scenario = scenario,
        rescale = True,
    )

Export v3.0.

In [ ]:
v30_path = os.path.join(paths['export'], "v3.0", str(year))
os.makedirs(v30_path, exist_ok=True)
db.to_txt(path = v30_path, scenario = scenario)

---
## Footprint comparison: v2.0 (legacy Electricity Maps) vs v3.0 (open ENTSO-E)

Non-EU / RoW regions are domestic-only in both pipelines, so their electricity footprints should be near-identical; any real change concentrates in the ENTSO-E-covered European countries whose import mix actually differs between the two datasets. Requires a `v2.0/<year>` export already on disk (`gen_v2.ipynb`).

In [ ]:
import pandas as pd

db_old = mario.parse_from_txt(
    path = os.path.join(paths['export'], "v2.0", str(year), "flows"),
    mode = "flows",
    table = 'SUT',
)

gwp = {
    "Carbon dioxide, fossil (air - Emiss)": 1.0,
    "CH4 (air - Emiss)": 25.0,
    "N2O (air - Emiss)": 298.0,
}

def ghg_footprint(f):
    f = f.loc[list(gwp), :].T
    return sum(f[substance] * factor for substance, factor in gwp.items())

f_v30 = ghg_footprint(db.query('f', scenarios=scenario))
f_old = ghg_footprint(db_old.f)

# Align the legacy pooled labels to the new naming convention.
item_renames = {}
for commodity in traded_commodities:
    pooled = db.meta.pooled_trade_map[commodity]
    item_renames[f"{commodity} supply"] = pooled['supply']
    item_renames[f"{commodity} need"] = pooled['need']
f_old = f_old.rename(index=item_renames, level='Item')

comp = pd.concat([f_old.rename('v2.0'), f_v30.rename('v3.0')], axis=1)
comp['Delta%'] = 100 * (comp['v3.0'] / comp['v2.0'] - 1)

In [ ]:
# Electricity-need GHG intensity per region, split EU (ENTSO-E-covered) vs non-EU.
NON_EU = {'US', 'CN', 'JP', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'ID', 'ZA',
          'CA', 'TW', 'WA', 'WL', 'WE', 'WF', 'WM'}
need = db.meta.pooled_trade_map['Electricity']['need']
ele = comp.loc[(slice(None), 'Commodity', need), :].copy()
ele.index = ele.index.get_level_values('Region')
ele['group'] = ['non-EU/RoW' if r in NON_EU else 'EU/ENTSO-E' for r in ele.index]

summary = ele.groupby('group')['Delta%'].agg(
    n='count',
    mean_abs=lambda s: s.abs().mean(),
    max_abs=lambda s: s.abs().max(),
).round(3)
print("Electricity-need footprint, v3.0 vs v2.0 — |Delta%| by group:")
print(summary)
print("\nLargest |Delta%| overall (electricity need):")
display(ele.reindex(ele['Delta%'].abs().sort_values(ascending=False).index).round(3).head(20))